# 10 - Expanded-data ingestion, cleaning and T-60 features (PySpark)

This notebook extends the previous pipeline without replacing it. It ingests
one canonical file per month, applies the same quality rules, audits null
behaviour and writes reusable Parquet.

Active task: arrival delay exactly 60 minutes before scheduled off-block.
Train runs through September 2022, validation is December 2022, test is March
2023 and the future test is June 2023. The two tests are never used to learn
imputers, category vocabularies, transformations or model settings.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from functools import reduce
from pyspark import StorageLevel
from pyspark.ml.feature import Imputer
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.flight_config import DataQualityConfig, FeatureConfig
from src.flight_data_catalog import (
    compare_null_cohorts, discover_monthly_flights,
    profile_nulls_by_month, validate_flight_schemas,
)
from src.spark_flight_pipeline import (
    apply_aviation_rules, build_value_transformer, calculate_delays,
    create_spark, fill_text_nulls, fit_frequent_categories,
    group_rare_categories, read_flights_spark,
    temporal_train_validation_two_test_split,
)
from src.t60_operational_features import add_prediction_cutoff, build_standard_t60_features

QUALITY = DataQualityConfig(min_delay_minutes=-120, regular_commercial_only=True)
FEATURES = FeatureConfig(
    target="Arrival_Delay_Min", prediction_horizon="pre_departure",
    apply_log=False, apply_yeo_johnson=False,
    categorical_hash_features=1 << 15, rare_category_min_count=1_000,
)
VALIDATION_START = "2022-12-01 00:00:00"
TEST_START = "2023-03-01 00:00:00"
FUTURE_TEST_START = "2023-06-01 00:00:00"
RUN_FULL_DATA = False
SMOKE_ROWS_PER_FILE = 2_000
SMOKE_SPLIT_ROWS = 100
WRITE_PARQUET = False
BUILD_OPERATIONAL_T60 = False
WINDOWS_HOURS = (1, 6, 24)
OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "expanded_arrival_pre_t60"
REPORT_ROOT = PROJECT_ROOT / "reports" / "expanded_data"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Canonical catalog, schema and null audit

Month folders take priority over legacy copies, preventing duplicate ingestion.
All 18 raw columns must match. Null rates in the three new files are compared
with the six original files. A two-percentage-point change is flagged for
review, but does not automatically remove a variable.

In [2]:
catalog = discover_monthly_flights(PROJECT_ROOT / "data" / "raw")
flight_files = [item.path for item in catalog]
catalog_pd = pd.DataFrame([
    {"month": item.month, "path": str(item.path), "source": item.source}
    for item in catalog
])
schema_audit = validate_flight_schemas(flight_files)
assert schema_audit["matches_reference_schema"].all(), schema_audit
null_profile = profile_nulls_by_month(
    flight_files,
    max_rows_per_file=None if RUN_FULL_DATA else SMOKE_ROWS_PER_FILE,
)
reference_months = ["202112", "202203", "202206", "202209", "202212", "202303"]
new_months = ["202106", "202109", "202306"]
null_comparison = compare_null_cohorts(
    null_profile, reference_months, new_months, material_delta_pp=2.0
)
catalog_pd.to_csv(REPORT_ROOT / "flight_file_catalog.csv", index=False)
schema_audit.to_csv(REPORT_ROOT / "schema_compatibility.csv", index=False)
null_profile.to_csv(REPORT_ROOT / "null_profile_by_month.csv", index=False)
null_comparison.to_csv(REPORT_ROOT / "null_profile_new_vs_reference.csv", index=False)
display(catalog_pd)
display(null_comparison)

,month,path,source
0,202106,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
1,202109,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
2,202112,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
3,202203,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
4,202206,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
5,202209,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
6,202212,C:\Users\celti\OneDrive - Universidade de Sant...,legacy_flights_folder
7,202303,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder
8,202306,C:\Users\celti\OneDrive - Universidade de Sant...,monthly_folder


,column,reference_null_pct,new_null_pct,delta_null_pp,material_change
0,AC Registration,0.025,0.116667,0.091667,False
1,AC Operator,0.000,0.000000,0.000000,False
2,AC Type,0.000,0.000000,0.000000,False
3,ACTUAL ARRIVAL TIME,0.000,0.000000,0.000000,False
4,ACTUAL OFF BLOCK TIME,0.000,0.000000,0.000000,False
5,ADEP,0.000,0.000000,0.000000,False
6,ADEP Latitude,0.000,0.000000,0.000000,False
7,ADEP Longitude,0.000,0.000000,0.000000,False
8,ADES,0.000,0.000000,0.000000,False
9,Actual Distance Flown (nm),0.000,0.000000,0.000000,False


## 2. PySpark ingestion and unchanged aviation rules

The explicit schema prevents month-dependent type inference. Rules remain:
scheduled flights only, delays no lower than -120 minutes, valid flight level,
positive distance and valid coordinates. Null predictors are retained for
train-only imputation.

In [3]:
spark = create_spark(
    "expanded-arrival-pre-cleaning", master="local[1]",
    driver_memory="2g", shuffle_partitions=8,
)
spark.sparkContext.setLogLevel("ERROR")
if RUN_FULL_DATA:
    raw = read_flights_spark(spark, [str(path) for path in flight_files])
else:
    monthly_smoke = [
        read_flights_spark(spark, str(path)).limit(SMOKE_ROWS_PER_FILE)
        for path in flight_files
    ]
    raw = reduce(lambda left, right: left.unionByName(right), monthly_smoke)
flights = calculate_delays(raw).withColumn(
    "Scheduled_Duration_Min",
    (F.col("FILED ARRIVAL TIME").cast("long")
     - F.col("FILED OFF BLOCK TIME").cast("long")) / F.lit(60.0),
)
before_rows = flights.count()
flights = apply_aviation_rules(flights, QUALITY)
flights = fill_text_nulls(
    flights, ["AC Operator", "AC Registration", "AC Type", "STATFOR Market Segment"]
).persist(StorageLevel.DISK_ONLY)
after_rows = flights.count()
print({"raw_rows": before_rows, "clean_rows": after_rows, "removed": before_rows-after_rows})

{'raw_rows': 18000, 'clean_rows': 16089, 'removed': 1911}


## 3. Four-way temporal split before fitted operations

December remains the selection period used previously. March is the first
untouched test; June is a later robustness test. Rows missing the arrival target
are counted separately and removed only after the split.

In [4]:
train, validation, test, future_test = temporal_train_validation_two_test_split(
    flights, VALIDATION_START, TEST_START, FUTURE_TEST_START
)
splits = {"train": train, "validation": validation, "test": test, "future_test": future_test}
missing_target = {}
for name, frame in list(splits.items()):
    missing_target[name] = frame.filter(F.col(FEATURES.target).isNull()).count()
    splits[name] = frame.filter(F.col(FEATURES.target).isNotNull())
split_counts = {name: frame.count() for name, frame in splits.items()}
assert all(value > 0 for value in split_counts.values()), split_counts
print({"split_counts": split_counts, "missing_target_removed": missing_target})

if not RUN_FULL_DATA:
    # Keep every temporal period represented while bounding all later actions.
    splits = {name: frame.limit(SMOKE_SPLIT_ROWS) for name, frame in splits.items()}
    split_counts = {name: frame.count() for name, frame in splits.items()}
    print({"bounded_smoke_split_counts": split_counts})

{'split_counts': {'train': 10612, 'validation': 1830, 'test': 1857, 'future_test': 1790}, 'missing_target_removed': {'train': 0, 'validation': 0, 'test': 0, 'future_test': 0}}


{'bounded_smoke_split_counts': {'train': 100, 'validation': 100, 'test': 100, 'future_test': 100}}


## 4. Train-only imputation, optional transforms and rare aircraft

Flight-level median, optional Yeo-Johnson lambdas and frequent aircraft types
are learned only on train. AC Type is not one-hot encoded: rare and unseen
values become OTHER. Log and Yeo-Johnson remain optional and disabled by
default.

In [5]:
value_transformer = build_value_transformer(splits["train"], FEATURES)
for name in splits:
    splits[name] = value_transformer.transform(splits[name])
imputer = Imputer(
    strategy="median", inputCols=["Requested FL"],
    outputCols=["Requested_FL_Imputed"],
).fit(splits["train"])
for name in splits:
    splits[name] = imputer.transform(splits[name])
frequent_ac_types = fit_frequent_categories(
    splits["train"], "AC Type", FEATURES.rare_category_min_count
)
for name in splits:
    splits[name] = group_rare_categories(
        splits[name], "AC Type", frequent_ac_types,
        output_column="AC Type_grouped",
    )
print({"frequent_ac_types": len(frequent_ac_types),
       "log": FEATURES.apply_log, "yeo_johnson": FEATURES.apply_yeo_johnson})

{'frequent_ac_types': 0, 'log': False, 'yeo_johnson': False}


## 5. T-60 feature gate and optional previous-flight variables

Base output contains only schedule and categorical predictors available at
T-60. If BUILD_OPERATIONAL_T60 is enabled, the same 1/6/24-hour airport, route,
operator and aircraft-rotation features are created. The reusable builder fails
if any event occurs after a target cutoff.

In [6]:
base_keep = [
    "ECTRL ID", FEATURES.target, "ADEP", "ADES", "AC Operator",
    "AC Registration", "AC Type_grouped", "STATFOR Market Segment",
    "FILED OFF BLOCK TIME", "FILED ARRIVAL TIME",
    "Requested_FL_Imputed", "Scheduled_Duration_Min",
]
model_splits = {}
for name, frame in splits.items():
    selected = frame.select(*[column for column in base_keep if column in frame.columns])
    model_splits[name] = add_prediction_cutoff(selected)
forbidden = {"ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
             "Departure_Delay_Min", "Actual Distance Flown (nm)"}
assert all(not (forbidden & set(frame.columns)) for frame in model_splits.values())

feature_audit = []
if BUILD_OPERATIONAL_T60:
    events = flights.select(
        "ECTRL ID", "ADEP", "ADES", "AC Operator", "AC Registration",
        "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
        "Departure_Delay_Min", "Arrival_Delay_Min",
    )
    for name in model_splits:
        model_splits[name], audit = build_standard_t60_features(
            model_splits[name], events, windows_hours=WINDOWS_HOURS
        )
        feature_audit.extend([{"split": name, **row} for row in audit])
feature_audit_pd = pd.DataFrame(feature_audit)
feature_audit_pd.to_csv(REPORT_ROOT / "t60_feature_leakage_audit.csv", index=False)
print({"operational_features": BUILD_OPERATIONAL_T60,
       "leakage_violations": int(feature_audit_pd.get(
           "leakage_violations", pd.Series(dtype=int)).sum())})

{'operational_features': False, 'leakage_violations': 0}


## 6. Post-clean null contract and optional Parquet output

Target nulls are removed and reported. Text uses explicit Unknown, flight level
uses the train median, and missing operational history is allowed when no prior
event exists. Set both RUN_FULL_DATA and WRITE_PARQUET to true for the complete
build. PySpark runs locally; no HDFS or external Hadoop cluster is used.

In [7]:
null_rows = []
for split_name, frame in model_splits.items():
    values = frame.agg(*[
        F.sum(F.col(column).isNull().cast("long")).alias(column)
        for column in frame.columns
    ]).first().asDict()
    total = split_counts[split_name]
    null_rows.extend([
        {"split": split_name, "column": column, "nulls": value,
         "rows": total, "null_pct": 100*value/total if total else None}
        for column, value in values.items()
    ])
post_clean_nulls = pd.DataFrame(null_rows)
post_clean_nulls.to_csv(REPORT_ROOT / "post_clean_null_profile.csv", index=False)
display(post_clean_nulls.sort_values(
    ["split", "null_pct"], ascending=[True, False]
).head(30))

if WRITE_PARQUET:
    for name, frame in model_splits.items():
        frame.write.mode("overwrite").parquet(str(OUTPUT_ROOT / name))
    pd.DataFrame([{
        "validation_start": VALIDATION_START, "test_start": TEST_START,
        "future_test_start": FUTURE_TEST_START,
        "operational_t60_built": BUILD_OPERATIONAL_T60,
        **{f"{name}_rows": count for name, count in split_counts.items()},
    }]).to_csv(REPORT_ROOT / "split_contract.csv", index=False)
else:
    print("Smoke complete. Enable full data and writes for notebook 11.")
flights.unpersist()
spark.stop()

,split,column,nulls,rows,null_pct
39,future_test,ECTRL ID,0,100,0.0
40,future_test,Arrival_Delay_Min,0,100,0.0
41,future_test,ADEP,0,100,0.0
42,future_test,ADES,0,100,0.0
43,future_test,AC Operator,0,100,0.0
44,future_test,AC Registration,0,100,0.0
45,future_test,AC Type_grouped,0,100,0.0
46,future_test,STATFOR Market Segment,0,100,0.0
47,future_test,FILED OFF BLOCK TIME,0,100,0.0
48,future_test,FILED ARRIVAL TIME,0,100,0.0


Smoke complete. Enable full data and writes for notebook 11.


## Decisions preserved

- Same scheduled-commercial population and physical limits.
- Same arrival-delay target and T-60 horizon.
- Same train-only fitting, rare aircraft OTHER and categorical hashing policy.
- Two earlier 2021 months expand train; March and June 2023 are two independent
  future evaluations.
- Raw and cleaned null behaviour is measured rather than assumed.